# Pandas Interview Study Summary

A practical, interview-focused notebook covering the main Pandas concepts in the **WeCreateProblems – Pandas Interview Questions and Answers** guide.

**Source:** https://www.wecreateproblems.com/interview-questions/pandas-interview-questions

The original guide is organised into beginner, intermediate and experienced-level questions. This notebook condenses those topics into a study path with explanations, code patterns, interview prompts and practical exercises rather than reproducing the source article verbatim.

> **Interview mindset:** be able to explain not only *what* a Pandas operation does, but also *why* you would use it, what it returns, common pitfalls, and how you would make it efficient on larger data.


## 1. Pandas mental model

Pandas is a Python library for working with labelled/tabular data.

### The two core structures

| Structure | Shape | Mental model |
|---|---:|---|
| `Series` | 1-D | A labelled column |
| `DataFrame` | 2-D | A labelled table made of Series |

Key ideas:

- **Index** → labels rows.
- **Columns** → labels variables.
- **Vectorized operations** → operate on whole columns/arrays rather than Python loops.
- **Alignment** → Pandas operations often align data by labels, not merely by position.
- **Missing data** → represented through Pandas/NumPy missing-value mechanisms.
- **Dtypes** → strongly affect correctness, memory usage and performance.


In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "age": [25, 31, 29, 42],
    "desk": ["FX", "Rates", "FX", "Equity"],
    "pnl": [1200.5, -350.0, 820.0, 2100.0]
})

df

## 2. Creating and inspecting DataFrames

### High-frequency interview commands

```python
pd.DataFrame(...)
pd.Series(...)
pd.read_csv(...)
pd.read_excel(...)

df.head()
df.tail()
df.shape
df.columns
df.index
df.dtypes
df.info()
df.describe()
df.nunique()
df["column"].value_counts()
```

### Interview distinction

- `df.shape` is an **attribute** → `(rows, columns)`.
- `df.info()` gives a structural summary, including non-null counts and dtypes.
- `df.describe()` gives descriptive statistics for numeric data by default.
- `df.head()` is a quick sanity check before doing analysis.


In [ ]:
print("shape:", df.shape)
print("columns:", list(df.columns))
print("dtypes:")
print(df.dtypes)
print("\nsummary:")
display(df.describe(numeric_only=True))

## 3. Selecting data: `[]`, `.loc` and `.iloc`

This is one of the most important interview areas.

### Column selection

```python
df["age"]                 # Series
df[["name", "age"]]       # DataFrame
```

### `.loc` — label-based selection

```python
df.loc[rows, columns]
df.loc[df["age"] > 30, ["name", "age"]]
```

### `.iloc` — integer-position-based selection

```python
df.iloc[0, 1]
df.iloc[:3, [0, 2]]
```

### `.at` and `.iat`

Use these for **single scalar values**:

```python
df.at[row_label, "age"]   # label based
df.iat[row_position, 1]  # integer-position based
```

**Interview trap:** `.loc` and `.iloc` are not interchangeable. `.loc` uses labels; `.iloc` uses integer positions.


In [ ]:
# Boolean filtering
df[df["age"] > 30]

# Multiple conditions: each condition needs parentheses
df[(df["age"] >= 25) & (df["desk"] == "FX")]

# Label-based selection
df.loc[df["pnl"] > 0, ["name", "pnl"]]

# Position-based selection
df.iloc[:2, :3]

## 4. Boolean filtering and `query()`

Typical patterns:

```python
df[df["pnl"] > 0]

df[(df["age"] > 25) & (df["desk"].isin(["FX", "Rates"]))]

df[df["name"].str.contains("a", case=False, na=False)]
```

`query()` can make complex filters more readable:

```python
df.query("age > 30 and pnl > 0")
```

### `loc` vs `query`

- `.loc` is explicit and flexible; it works naturally with arbitrary Python expressions.
- `query()` provides a compact expression syntax and can be convenient for readable filtering.


## 5. Missing values

Core operations:

```python
df.isna()
df.isna().sum()

df.dropna()
df.dropna(subset=["age"])

df.fillna(0)
df["age"].fillna(df["age"].median())

df.ffill()
df.bfill()
```

### Interview considerations

Do not automatically replace every missing value with zero. Ask:

1. What does missing mean in this dataset?
2. Is the variable numeric, categorical or time-series?
3. Would deletion introduce bias?
4. Should imputation use mean, median, a business rule, forward fill, etc.?
5. Should missingness itself become a feature?

For time-series data, `ffill`/`bfill` must be justified by the meaning and ordering of the data.


## 6. Dtypes and type conversion

Useful tools:

```python
df.dtypes
df["age"].astype("int64")
pd.to_numeric(...)
pd.to_datetime(...)
df.convert_dtypes()
```

### `astype()` vs `convert_dtypes()`

- `astype()` explicitly requests a dtype conversion.
- `convert_dtypes()` attempts to choose Pandas' more appropriate nullable dtypes.

Be careful with numeric conversion of dirty strings:

```python
pd.to_numeric(series, errors="coerce")
```

`errors="coerce"` turns invalid values into missing values, which should then be handled deliberately.


## 7. Sorting, duplicates and index management

### Sorting

```python
df.sort_values("pnl")
df.sort_values(["desk", "pnl"], ascending=[True, False])
df.sort_index()
```

### Duplicates

```python
df.duplicated()
df.duplicated(subset=["name"])
df.drop_duplicates()
```

### Index

```python
df.set_index("name")
df.reset_index()
```

Remember that an index is part of the DataFrame's structure; it is not necessarily a business key.

**Interview question:** What is the difference between `reset_index()` and dropping a column?  
`reset_index()` moves index information into columns (unless configured otherwise), whereas dropping a column removes an ordinary column from the DataFrame.


## 8. GroupBy and aggregation

The standard mental model is:

> **split → apply → combine**

Example:

```python
df.groupby("desk")["pnl"].sum()
```

Multiple aggregations:

```python
df.groupby("desk").agg(
    pnl_sum=("pnl", "sum"),
    pnl_mean=("pnl", "mean"),
    trades=("pnl", "size")
)
```

### `agg()` vs `transform()`

`agg()` generally reduces each group to one or more summary rows.

`transform()` returns a result aligned to the original rows:

```python
df["desk_mean_pnl"] = (
    df.groupby("desk")["pnl"].transform("mean")
)
```

This makes `transform()` especially useful for group-level features such as deviations from a group mean.


In [ ]:
group_summary = (
    df.groupby("desk")
      .agg(
          trades=("pnl", "size"),
          total_pnl=("pnl", "sum"),
          avg_pnl=("pnl", "mean")
      )
      .sort_values("total_pnl", ascending=False)
)

group_summary

## 9. Combining DataFrames: `concat`, `merge`, `join`

### `concat()`

Think **stack/concatenate**:

```python
pd.concat([df1, df2], axis=0)  # rows
pd.concat([df1, df2], axis=1)  # columns
```

### `merge()`

Think **relational/database join**:

```python
pd.merge(left, right, on="trade_id", how="left")
```

Common join types:

- `inner`
- `left`
- `right`
- `outer`

### `join()`

A convenient method primarily designed around joining using indexes (although it can also join on a column).

**Interview rule of thumb:** if the question sounds like a SQL join on a key, think `merge()`; if the question is about stacking compatible tables, think `concat()`.


## 10. Reshaping: `pivot`, `pivot_table`, `melt`

### `pivot()`

Reshapes data without aggregation and requires combinations to be unique.

```python
df.pivot(index="date", columns="desk", values="pnl")
```

### `pivot_table()`

Like a spreadsheet pivot table; it can aggregate duplicate combinations:

```python
df.pivot_table(
    index="desk",
    values="pnl",
    aggfunc="sum"
)
```

### `melt()`

Converts wide data into long/tidy format:

```python
pd.melt(
    wide_df,
    id_vars=["id"],
    var_name="metric",
    value_name="value"
)
```

**Memory aid:** `pivot` makes data wider; `melt` makes data longer.


## 11. String operations

Pandas provides vectorized string operations through `.str`:

```python
df["name"].str.lower()
df["name"].str.upper()
df["name"].str.strip()

df["name"].str.contains("a", case=False, na=False)
df["name"].str.startswith("A")
df["name"].str.replace("old", "new", regex=False)

df["name"].str.len()
df["name"].str.split()
```

Avoid Python loops when a vectorized Pandas/string operation expresses the same transformation.


## 12. Dates and time series

Core tools:

```python
df["date"] = pd.to_datetime(df["date"])

df = df.set_index("date")

df.resample("D").sum()
df.resample("M").mean()

df["rolling_mean"] = df["pnl"].rolling(20).mean()
df["expanding_mean"] = df["pnl"].expanding().mean()
```

### Concepts to explain in an interview

- **DatetimeIndex** enables time-aware indexing and resampling.
- **Resampling** changes the time frequency, e.g. daily → monthly.
- **Rolling window** calculates statistics over a moving window.
- **Expanding window** uses all observations available up to each point.
- **Timezone-aware timestamps** are important when combining data from multiple markets or regions.


## 13. `apply`, `map`, `transform` and vectorization

### `map()`

Usually applies a mapping/function element-wise to a Series:

```python
df["desk_code"] = df["desk"].map({
    "FX": 1,
    "Rates": 2,
    "Equity": 3
})
```

### `apply()`

Flexible, but can be slower than vectorized operations:

```python
df["pnl_abs"] = df["pnl"].apply(abs)
```

### Prefer vectorization

Instead of:

```python
df["pnl_abs"] = df["pnl"].apply(lambda x: abs(x))
```

prefer:

```python
df["pnl_abs"] = df["pnl"].abs()
```

**Interview point:** `apply()` is useful when no suitable vectorized operation exists, but it should not be the automatic solution for every transformation.


## 14. Categorical data and memory

Categorical data is useful when a column contains a relatively small set of repeated values.

```python
df["desk"] = df["desk"].astype("category")
```

Benefits can include lower memory usage and faster operations for appropriate workloads.

Inspect memory:

```python
df.info(memory_usage="deep")
df.memory_usage(deep=True)
```

For large datasets, also consider:

- loading only required columns with `usecols=...`
- choosing appropriate numeric dtypes
- categorical encoding for low-cardinality strings
- processing data in chunks
- avoiding unnecessary copies
- filtering early
- using vectorized operations


## 15. Large datasets and performance

### First principles

1. **Measure before optimizing.**
2. Avoid unnecessary Python-level loops.
3. Prefer vectorized operations.
4. Reduce columns and rows as early as possible.
5. Use suitable dtypes.
6. Process files in chunks when the complete dataset does not fit memory.
7. Consider a database or distributed framework when Pandas is no longer the right execution engine.

Example:

```python
for chunk in pd.read_csv("large.csv", chunksize=100_000):
    process(chunk)
```

For distributed workloads, tools such as Dask or Ray can be considered depending on the workload and architecture.

### Benchmarking

```python
%timeit df["pnl"].mean()
```

For larger investigations, use profiling tools rather than guessing where the bottleneck is.


## 16. Financial-data patterns

For quant/data-science interviews, connect Pandas concepts to real workflows.

### Returns

```python
prices["return"] = prices["close"].pct_change()
```

### Rolling volatility

```python
prices["vol_20d"] = (
    prices["return"]
    .rolling(20)
    .std()
)
```

### Grouped statistics

```python
trades.groupby("desk").agg(
    pnl=("pnl", "sum"),
    volume=("notional", "sum")
)
```

### Time alignment

When combining market data with events, make timestamps explicit, sorted and timezone-consistent before performing time-based joins or resampling.


## 17. MultiIndex

A `MultiIndex` represents hierarchical labels on one or more axes.

Example:

```python
multi = df.set_index(["desk", "name"])
```

Useful operations:

```python
multi.loc[("FX", "Alice")]
multi.reset_index()
multi.sort_index()
```

Typical interview use cases include hierarchical financial data, panel-like structures and grouped results.

**Pitfall:** MultiIndex can make code harder to reason about. Resetting the index can often make downstream transformations easier to read.


## 18. Common interview traps

### `and` vs `&`

Wrong:

```python
df[(df["age"] > 25) and (df["pnl"] > 0)]
```

Correct:

```python
df[(df["age"] > 25) & (df["pnl"] > 0)]
```

Use `&`, `|`, and `~` for element-wise boolean operations, with parentheses around conditions.

### `merge` vs `concat`

- `merge` → relational/key-based combination.
- `concat` → append/stack along an axis.

### `loc` vs `iloc`

- `loc` → labels.
- `iloc` → integer positions.

### `pivot` vs `pivot_table`

- `pivot` → no aggregation; combinations need to be unique.
- `pivot_table` → supports aggregation.

### `agg` vs `transform`

- `agg` → summary.
- `transform` → result aligned to original rows.

### `apply` vs vectorization

Prefer native/vectorized operations when available.


## 19. Rapid interview cheat sheet

| Task | Typical Pandas tool |
|---|---|
| Load CSV | `pd.read_csv()` |
| Load Excel | `pd.read_excel()` |
| Inspect first rows | `head()` |
| Inspect structure | `info()` |
| Descriptive stats | `describe()` |
| Select column | `df["col"]` |
| Select by label | `.loc[]` |
| Select by position | `.iloc[]` |
| Single scalar by label | `.at[]` |
| Single scalar by position | `.iat[]` |
| Filter rows | Boolean indexing / `query()` |
| Missing values | `isna()`, `fillna()`, `dropna()` |
| Convert dtype | `astype()`, `convert_dtypes()` |
| Parse dates | `pd.to_datetime()` |
| Sort | `sort_values()` / `sort_index()` |
| Remove duplicates | `drop_duplicates()` |
| Group | `groupby()` |
| Aggregate | `agg()` |
| Group-aligned feature | `transform()` |
| Stack tables | `concat()` |
| Key-based join | `merge()` |
| Reshape wide | `pivot()` / `pivot_table()` |
| Reshape long | `melt()` |
| String operations | `.str` |
| Time frequency | `resample()` |
| Moving window | `rolling()` |
| Running window | `expanding()` |
| Cumulative result | `cumsum()`, `cumprod()` |
| Memory inspection | `memory_usage()` / `info()` |


## 20. Interview questions to practise

Try answering these **without looking at the previous sections**.

### Beginner
1. What is the difference between a Series and a DataFrame?
2. Explain `.loc` versus `.iloc`.
3. How would you inspect a new DataFrame?
4. How do you detect and handle missing values?
5. How do you remove duplicates?
6. How do you convert a column to datetime?
7. How do you filter using multiple conditions?
8. What does `groupby()` do?
9. What is the difference between `merge()` and `concat()`?
10. How do you select several columns?

### Intermediate
11. Explain `agg()` versus `transform()`.
12. When would you use `pivot_table()`?
13. How do `pivot()` and `melt()` differ?
14. How would you calculate a rolling mean?
15. How would you work with categorical data?
16. What is a MultiIndex?
17. How do you perform a left join?
18. What is the difference between `apply()` and vectorized operations?
19. How do you resample time-series data?
20. How would you handle timezone-aware timestamps?

### Experienced
21. How would you diagnose a slow Pandas pipeline?
22. How would you reduce the memory footprint of a large DataFrame?
23. What would you do if a CSV does not fit into memory?
24. When would you stop using Pandas and move to a database/distributed framework?
25. How would you design a robust time-series preprocessing pipeline?
26. What are the performance implications of `apply()`?
27. How would you profile a large dataset?
28. How would you handle irregular timestamps?
29. How would you make a group-wise feature without losing row alignment?
30. Explain vectorization and why it matters.


## 21. Practical coding exercises

### Exercise A — Cleaning

Create a DataFrame containing:

- duplicate rows
- missing numeric values
- missing categorical values
- numeric values stored as strings

Then clean it and explain each decision.

### Exercise B — Group analytics

Given a trades DataFrame with:

`trade_id`, `desk`, `instrument`, `notional`, `pnl`, `timestamp`

produce:

1. total P&L by desk
2. average trade P&L by instrument
3. trade count by desk
4. desk-level P&L as a percentage of total P&L
5. a column containing each trade's deviation from its desk's mean P&L

### Exercise C — Time series

Given daily prices:

1. calculate returns
2. calculate a 20-day rolling volatility
3. calculate monthly returns
4. identify missing trading dates
5. explain how you would handle timezone information

### Exercise D — Performance

Take a DataFrame with several million rows and investigate:

- memory usage
- expensive operations
- object/string columns
- opportunities for categorical encoding
- unnecessary Python loops

Write down what you would measure before changing the implementation.


## 22. Final interview framework

When asked a Pandas question, structure your answer:

**1. Define it**  
What does the operation/concept mean?

**2. Show the idiomatic syntax**  
Give a short, readable example.

**3. Explain the output**  
What type/shape/alignment should the interviewer expect?

**4. Mention the pitfall**  
For example: label vs position, missing values, duplicate keys, or `apply()` performance.

**5. Scale it**  
If the dataset is large, discuss memory, vectorization, chunking, profiling and whether Pandas remains appropriate.

### The five concepts worth knowing cold

1. `.loc` vs `.iloc`
2. `groupby()` + `agg()` vs `transform()`
3. `merge()` vs `concat()`
4. vectorization vs `apply()`
5. time-series operations: `to_datetime()` → index → `resample()` / `rolling()`

These concepts recur across a large proportion of practical Pandas interview problems.


## Source and scope

This notebook was created from the topic structure and explanations in the WeCreateProblems **“100+ Pandas Interview Questions and Answers”** page, which covers beginner, intermediate and experienced Pandas questions including DataFrames/Series, selection, missing data, grouping, joins, reshaping, time series and performance. citeturn0view0

The notebook intentionally **summarises and restructures** the material into original study notes and examples rather than copying the source article's full question-and-answer text.

Source: https://www.wecreateproblems.com/interview-questions/pandas-interview-questions
